In [ ]:
# %% cell 0
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# Data directory
data_dir = r"C:\Users\Asus\OneDrive\hanif\Kuliah\UGM\Semester 5\Artificial Intelligence\UTS\Tugas\Data"
if not os.path.exists(data_dir):
    data_dir = os.path.abspath(os.path.join("..", "Data"))

print(f"Using data directory: {data_dir}")
for dirname, _, filenames in os.walk(data_dir):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# %% cell 1
pscd = os.path.join(data_dir, "ALL_FAIL_LOG_PSCD.csv")
gps  = os.path.join(data_dir, "ALL_FAIL_LOG_GPS_0 (1).csv")
mag  = os.path.join(data_dir, "ALL_FAIL_LOG_MAG_0.csv")
bat  = os.path.join(data_dir, "ALL_FAIL_LOG_BAT_0.csv")
rate = os.path.join(data_dir, "ALL_FAIL_LOG_RATE.csv")
ctun = os.path.join(data_dir, "ALL_FAIL_LOG_CTUN.csv")
motb = os.path.join(data_dir, "ALL_FAIL_LOG_MOTB.csv")

# %% cell 2
pscd_df = pd.read_csv(pscd)
print(f"shape of the pscd_df: {pscd_df.shape}, \nfeatures in pscd_df: {list(pscd_df.columns)}")

gps_df = pd.read_csv(gps)
print(f"\nshape of the gps_df: {gps_df.shape}, \nfeatures in gps_df: {list(gps_df.columns)}")

mag_df = pd.read_csv(mag)
print(f"\nshape of the mag_df: {mag_df.shape}, \nfeatures in mag_df: {list(mag_df.columns)}")

# %% cell 3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    accuracy_score,
    f1_score,
)
from xgboost import XGBClassifier

# 1️⃣ Load / Prepare Data
df = gps_df.copy()

# Drop metadata, timestamps, and flight path spatial coordinates to prevent data leakage
# (GMS is a continuous GPS timestamp; Lat/Lng/Alt encode fixed trajectory positions)
leakage_cols = ['LineNo', 'TimeUS', 'GMS', 'GWk', 'I', 'Lat', 'Lng', 'Alt']
df = df.drop(columns=leakage_cols, errors='ignore')

# Separate features and labels
X = df.drop(columns=['labels'])
y = df['labels']

# 2️⃣ Correlation Analysis
plt.figure(figsize=(10, 8))
corr = X.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title("Feature Correlation Heatmap (GPS Data)")
plt.tight_layout()
plt.show()

# Optionally remove highly correlated features (corr > 0.9)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
if to_drop:
    print(f"Dropping {len(to_drop)} highly correlated features:", to_drop)
    X = X.drop(columns=to_drop, errors='ignore')
else:
    print("No features with correlation > 0.9 found. Keeping all features:", list(X.columns))

# 3️⃣ Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Impute missing values fitted ONLY on train set to prevent leakage
train_median = X_train.median()
X_train = X_train.fillna(train_median)
X_test = X_test.fillna(train_median)

# 4️⃣ Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5️⃣ Train Boosting Model (XGBoost) with validation tracking
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)
xgb.fit(
    X_train_scaled, y_train,
    eval_set=[(X_train_scaled, y_train), (X_test_scaled, y_test)],
    verbose=False
)

# 6️⃣ Predictions & Probabilities
y_train_pred = xgb.predict(X_train_scaled)
y_test_pred = xgb.predict(X_test_scaled)
y_train_proba = xgb.predict_proba(X_train_scaled)
y_test_proba = xgb.predict_proba(X_test_scaled)

# 7️⃣ Overfitting / Underfitting Check
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
train_f1 = f1_score(y_train, y_train_pred, average='macro')
test_f1 = f1_score(y_test, y_test_pred, average='macro')
train_auc = roc_auc_score(y_train, y_train_proba, multi_class='ovr', average='macro')
test_auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')

print("\n" + "=" * 55)
print("       OVERFITTING / UNDERFITTING DIAGNOSTIC")
print("=" * 55)
print(f"{'Metric':<18} {'Train':<10} {'Test':<10} {'Gap (Train - Test)':<20}")
print("-" * 55)
print(f"{'Accuracy':<18} {train_acc:<10.4f} {test_acc:<10.4f} {train_acc - test_acc:<20.4f}")
print(f"{'Macro F1':<18} {train_f1:<10.4f} {test_f1:<10.4f} {train_f1 - test_f1:<20.4f}")
print(f"{'Macro ROC-AUC':<18} {train_auc:<10.4f} {test_auc:<10.4f} {train_auc - test_auc:<20.4f}")
print("=" * 55)

if (train_acc - test_acc) > 0.15:
    print("Diagnosis: Model shows signs of OVERFITTING (large train-test gap).")
elif train_acc < 0.70 and test_acc < 0.70:
    print("Diagnosis: Model shows signs of UNDERFITTING (low performance on both).")
else:
    print("Diagnosis: Model has a HEALTHY FIT with high generalization capability.")

# Test Classification Report & Confusion Matrix
print("\nClassification Report (Test Set):\n", classification_report(y_test, y_test_pred))
print("Confusion Matrix (Test Set):\n", confusion_matrix(y_test, y_test_pred))

# 8️⃣ ROC & AUC Curves (One-vs-Rest for Multi-Class)
classes = np.unique(y)
y_test_bin = label_binarize(y_test, classes=classes)

class_names = {
    0: "0: Normal",
    1: "1: GPS Anomaly",
    2: "2: ACC Anomaly",
    3: "3: Engine Anomaly",
    4: "4: RC Anomaly"
}

fpr = dict()
tpr = dict()
roc_auc = dict()

for i, c in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_test_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average ROC
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_test_proba.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Plot ROC Curves
plt.figure(figsize=(10, 8))
colors = ['blue', 'green', 'red', 'purple', 'orange']
for i, c in enumerate(classes):
    label_text = f"{class_names.get(c, f'Class {c}')} (AUC = {roc_auc[i]:.4f})"
    plt.plot(fpr[i], tpr[i], color=colors[i % len(colors)], lw=2, label=label_text)

plt.plot(fpr["micro"], tpr["micro"], color='deeppink', linestyle=':', lw=3,
         label=f'Micro-average (AUC = {roc_auc["micro"]:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (FPR)", fontsize=12)
plt.ylabel("True Positive Rate (TPR)", fontsize=12)
plt.title("Multi-Class ROC Curves (One-vs-Rest on Test Set)", fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 9️⃣ Learning Curve (Train vs Test Loss over Boosting Iterations)
evals_result = xgb.evals_result()
train_loss = evals_result['validation_0']['mlogloss']
test_loss = evals_result['validation_1']['mlogloss']

plt.figure(figsize=(10, 6))
plt.plot(train_loss, label='Train Logloss', color='blue', lw=2)
plt.plot(test_loss, label='Test Logloss (Validation)', color='orange', lw=2)
plt.axvline(x=np.argmin(test_loss), color='green', linestyle='--', label=f'Best Test Loss (Iter {np.argmin(test_loss)+1})')
plt.xlabel("Boosting Round (Trees)", fontsize=12)
plt.ylabel("Multi-class Logloss", fontsize=12)
plt.title("XGBoost Learning Curve: Overfitting / Underfitting Check", fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 🔟 Feature Importance
plt.figure(figsize=(10, 6))
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]
sns.barplot(y=X.columns[indices], x=importances[indices], orient='h')
plt.title("Feature Importances (XGBoost)", fontsize=14)
plt.xlabel("Importance Score", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()
plt.show()


# %% cell 4


